In [2]:
import numpy as np
import torch
from torch import concat, diag, logical_and, logical_or, nn, tensor, tile
from torch.nn import Dropout
import pytorch_lightning as py_light
from functools import partial

# Các class, hàm cần thiết trong code cũ

In [3]:
def multiply_head_with_embedding(prediction_head, embeddings):
    return prediction_head.matmul(embeddings.transpose(-1, -2))

In [4]:
def bce_loss(pos_logits, neg_logits, mask, epsilon=1e-10, device="cpu"):
    loss = log(1. + exp(-pos_logits) + epsilon) + log(1. + exp(neg_logits) + epsilon).mean(-1, keepdim=True)
    return (loss * mask.unsqueeze(-1)).sum() / mask.sum()

In [5]:
class DynamicPositionEmbedding(torch.nn.Module):

    def __init__(self, max_len, dimension):
        super(DynamicPositionEmbedding, self).__init__()
        self.max_len = max_len
        self.embedding = nn.Embedding(max_len, dimension)
        self.pos_indices = torch.arange(0, self.max_len, dtype=torch.int)
        self.register_buffer('pos_indices_const', self.pos_indices)

    def forward(self, x, device='cpu'):
        seq_len = x.shape[1]
        return self.embedding(self.pos_indices_const[-seq_len:]) + x


In [6]:
class SASRec(py_light.LightningModule):

    def __init__(self,
                 hidden_size,
                 dropout_rate,
                 max_len,
                 num_items,
                 batch_size,
                 sampling_style,
                 topk_sampling=False,
                 topk_sampling_k=1000,
                 learning_rate=0.001,
                 num_layers=2,
                 loss='bce',
                 bpr_penalty=None,
                 optimizer='adam',
                 output_bias=False,
                 share_embeddings=True,
                 final_activation=False):
        super(SASRec, self).__init__()
        self.learning_rate = learning_rate
        self.hidden_size = hidden_size
        self.dropout_rate = dropout_rate
        self.num_items = num_items
        self.batch_size = batch_size
        self.num_layers = num_layers
        self.max_len = max_len
        self.output_bias = output_bias
        self.share_embeddings = share_embeddings
        self.future_mask = torch.triu(torch.ones(max_len, max_len) * float('-inf'), diagonal=1)
        self.register_buffer('future_mask_const', self.future_mask)
        self.register_buffer('seq_diag_const', ~diag(torch.ones(max_len, dtype=torch.bool)))
        self.register_buffer('bias_ones', torch.ones([self.batch_size, self.max_len, 1]))
        if output_bias and share_embeddings:
            self.item_embedding = nn.Embedding(num_items + 1, hidden_size + 1, padding_idx=0)
        else:
            self.item_embedding = nn.Embedding(num_items + 1, hidden_size, padding_idx=0)
        self.positional_embedding_layer = DynamicPositionEmbedding(max_len, hidden_size)

        torch.nn.init.xavier_uniform_(self.item_embedding.weight.data)
        torch.nn.init.xavier_uniform_(self.positional_embedding_layer.embedding.weight.data)

        if share_embeddings:
            self.output_embedding = self.item_embedding
        elif (not share_embeddings) and output_bias:
            self.output_embedding = nn.Embedding(num_items + 1, hidden_size + 1, padding_idx=0)
        else:
            self.output_embedding = nn.Embedding(num_items + 1, hidden_size, padding_idx=0)

        self.norm = nn.LayerNorm([hidden_size])
        self.input_dropout = Dropout(dropout_rate)
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_size,
                                                   nhead=1,
                                                   dim_feedforward=hidden_size,
                                                   dropout=dropout_rate,
                                                   batch_first=True,
                                                   norm_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=self.num_layers, norm=self.norm)
        self.merge_attn_mask = True
        if final_activation:
            self.final_activation = nn.ELU(0.5)
        else:
            self.final_activation = nn.Identity()

        self.loss_fn = loss
        if self.loss_fn == 'bce':
            self.loss = bce_loss
        elif self.loss_fn == 'ssm':
            self.loss = sampled_softmax_loss
        elif self.loss_fn == 'bpr-max':
            if bpr_penalty is not None:
                self.loss = partial(bpr_max_loss, bpr_penalty)
            else:
                raise ValueError('bpr_penalty must be provided for bpr_max loss')
        else:
            raise ValueError('Loss function not supported')

        self.sampling_style = sampling_style
        self.topk_sampling = topk_sampling
        self.topk_sampling_k = topk_sampling_k
        self.optimizer = optimizer
        self.save_hyperparameters()

    def merge_attn_masks(self, padding_mask):
        batch_size = padding_mask.shape[0]
        seq_len = padding_mask.shape[1]

        if not self.merge_attn_mask:
            return self.future_mask_const[:seq_len, :seq_len]

        padding_mask_broadcast = ~padding_mask.bool().unsqueeze(1)
        future_masks = tile(self.future_mask_const[:seq_len, :seq_len], (batch_size, 1, 1))
        merged_masks = logical_or(padding_mask_broadcast, future_masks)
        # Always allow self-attention to prevent NaN loss
        # See: https://github.com/pytorch/pytorch/issues/41508
        diag_masks = tile(self.seq_diag_const[:seq_len, :seq_len], (batch_size, 1, 1))
        return logical_and(diag_masks, merged_masks)

    def forward(self, item_indices, mask):
        att_mask = self.merge_attn_masks(mask)
        items = self.item_embedding(
            item_indices)[:, :, :-1] if self.output_bias and self.share_embeddings else self.item_embedding(item_indices)
        x = items * np.sqrt(self.hidden_size)
        x = self.positional_embedding_layer(x)
        x = self.encoder(self.input_dropout(x), att_mask)
        return concat([x, self.bias_ones], dim=-1) if self.output_bias else x

    def training_step(self, batch, _):
        x_hat = self.forward(batch["clicks"], batch["mask"])
        train_loss = calc_loss(self.loss, x_hat, batch["labels"], batch["uniform_negatives"], batch["in_batch_negatives"],
                               batch["mask"], self.output_embedding, self.sampling_style, self.final_activation,
                               self.topk_sampling, self.topk_sampling_k, self.device)
        self.log("train_loss", train_loss)
        return train_loss

    def validation_step(self, batch, _batch_idx):
        x_hat = self.forward(batch['clicks'], batch['mask'])
        cut_offs = tensor([5, 10, 20], device=self.device)
        recall, mrr = validate_batch_per_timestamp(batch, x_hat, self.output_embedding, cut_offs)
        test_loss = calc_loss(self.loss, x_hat, batch["labels"], batch["uniform_negatives"], batch["in_batch_negatives"],
                              batch["mask"], self.output_embedding, self.sampling_style, self.final_activation,
                              self.topk_sampling, self.topk_sampling_k, self.device)
        for i, k in enumerate(cut_offs.tolist()):
            self.log(f'recall_cutoff_{k}', recall[i])
            self.log(f'mrr_cutoff_{k}', mrr[i])
        self.log('test_seq_len', x_hat.shape[1])
        self.log('test_loss', test_loss)

    def configure_optimizers(self):
        if self.optimizer == 'adam':
            optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        elif self.optimizer == 'adagrad':
            optimizer = torch.optim.Adagrad(self.parameters(), lr=self.learning_rate)
        else:
            raise ValueError('Optimizer not supported, please use adam or adagrad')
        return optimizer

    def export_topk_onnx(self, out_dir):
        top_k_model = TopKModel(self)
        top_k_model.export_onnx(out_dir)

    def export(self, out_dir):
        self.export_topk_onnx(out_dir)

# inference

In [7]:
class SASRecInference:
    def __init__(self, checkpoint_path, device = None):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        self.model = SASRec.load_from_checkpoint(checkpoint_path, map_location=device)
        self.model.eval()
        self.model.to(device)
        self.max_len = self.model.max_len
        self.num_items = self.model.num_items

    def _prepare_sequence(self, click_sequence: list[int]) -> tuple[torch.Tensor, torch.Tensor]:
        # Chuyển chuỗi aid (đã +1) thành tensor đã pad phù hợp với model
        # Returns:
        #     item_indices: (1, max_len) — tensor đã left-pad bằng 0
        #     mask        : (1, max_len) — 1 = vị trí thực, 0 = pad
        
        # Chỉ lấy max_len item cuối (giống lúc train)
        seq = click_sequence[-self.max_len:]
        seq_len = len(seq)
 
        pad_len = self.max_len - seq_len
        padded = [0] * pad_len + seq                         # left-pad
        mask_arr = [0.0] * pad_len + [1.0] * seq_len
 
        item_indices = torch.tensor([padded], dtype=torch.long, device=self.device)
        mask = torch.tensor([mask_arr], dtype=torch.float, device=self.device)
        return item_indices, mask

    #  top-K item
    @torch.no_grad()
    def recommend_topk(self, click_sequence: list[int], k = 20, exclude_clicked: bool = True) -> tuple[list[int], list[float]]:
        #Gợi ý top-K item tiếp theo cho chuỗi click của 1 user.
        #Returns:
            # top_aids   : list k aid (đã -1, về aid gốc của OTTO)
            # top_scores : list k điểm tương ứng
       
        item_indices, mask = self._prepare_sequence(click_sequence)
 
        # Forward pass — output: (1, max_len, hidden_size)
        x_hat = self.model.forward(item_indices, mask)
 
        # Lấy hidden state ở vị trí cuối cùng (next-item prediction)
        last_hidden = x_hat[:, -1, :]  # shape: (1, hidden_size)
 
        # Tính logit với toàn bộ embedding bảng
        # output_embedding.weight: (num_items+1, hidden_size)
        logits = multiply_head_with_embedding(
            last_hidden,
            self.model.output_embedding.weight,
        )  # shape: (1, num_items+1)
 
        logits = logits.squeeze(0)  # (num_items+1,)
 
        # Đặt điểm padding item (idx=0) về -inf
        logits[0] = float("-inf")
 
        # Loại bỏ item đã click nếu cần
        if exclude_clicked:
            for aid in click_sequence:
                if 1 <= aid <= self.num_items:
                    logits[aid] = float("-inf")
 
        # Lấy top-K
        actual_k = min(k, self.num_items)
        top_scores_tensor, top_indices_tensor = torch.topk(logits, k=actual_k)
        top_indices_tensor, top_scores_tensor = top_indices_tensor, top_scores_tensor
        top_aids_shifted = top_indices_tensor.cpu().tolist()
        top_scores = top_scores_tensor.cpu().tolist()
 
        # Chuyển về aid gốc của OTTO (trừ 1 vì lúc train đã +1)
        top_aids = [aid - 1 for aid in top_aids_shifted]
 
        return top_aids, top_scores


In [8]:
CHECKPOINT_PATH = "/kaggle/input/models/sandaria/sasrec/pytorch/otto-dataset/1/sasrec-otto-epoch4-recall_cutoff_200.308.ckpt"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [9]:
engine = SASRecInference(CHECKPOINT_PATH, device=DEVICE)

/tmp/ipykernel_57/3872232286.py:58: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=self.num_layers, norm=self.norm)


[INFO] Model loaded from: /kaggle/input/models/sandaria/sasrec/pytorch/otto-dataset/1/sasrec-otto-epoch4-recall_cutoff_200.308.ckpt
[INFO] Device         : cuda
[INFO] max_len        : 50
[INFO] num_items      : 1855603
[INFO] hidden_size    : 200


In [11]:
sample_aid_goc = [1492293,910862,1491172,424964,1515526,440486,109488,1507622,1734061,854637,718983,215311,718983,711125,50049,105393,959544,1734061,1842593,1464360,207905,1628317]
sample_aid_updated = [id + 1 for id in sample_aid_goc]

top_aids, top_scores = engine.recommend_topk(
        click_sequence=sample_aid_updated,
        k=20,
        exclude_clicked=True,
    )
print(f"Input clicks: {sample_aid_goc}")
print(f"Top-20: {top_aids}")
print(f"Scores: {[f'{s:.4f}' for s in top_scores[:5]]} ...")376932


=== Single Session Inference ===
Input clicks: [1492293, 910862, 1491172, 424964, 1515526, 440486, 109488, 1507622, 1734061, 854637, 718983, 215311, 718983, 711125, 50049, 105393, 959544, 1734061, 1842593, 1464360, 207905, 1628317]
Top-20 recommended aids (gốc): [556314, 1123537, 464673, 1007803, 1763549, 103974, 880854, 1491502, 1453277, 937893, 165854, 259724, 1245311, 606921, 1781681, 1532558, 434686, 1568162, 924095, 1483294]
Scores: ['14.0753', '13.9654', '13.2728', '13.0544', '12.9465'] ...
